In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Load the dataset
df = pd.read_csv("ALS_progression_rate.csv")

# Preview the data
print(df.columns)

# Make sure your target column is correct — assuming it's called 'dFRS'
# If not, replace 'dFRS' with the actual target column name
df = df.dropna(subset=["dFRS"])
X = df.drop(columns=["dFRS"])
y = df["dFRS"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a regression model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)

# Save predictions
predictions_df = pd.DataFrame({"Prediction": y_pred})
predictions_df.to_csv("ALS_predictions.csv", index=False)
print("Predictions saved to ALS_predictions.csv")


Index(['dFRS', 'Onset.Delta', 'Symptom.Speech', 'Symptom.WEAKNESS',
       'Symptom.OTHER', 'Symptom.Swallowing', 'Symptom.GAIT_CHANGES',
       'Symptom.Atrophy', 'Symptom.Cramps', 'Symptom.Fasciculations',
       ...
       'max.slope.bp.systolic', 'min.slope.bp.systolic',
       'last.slope.bp.systolic', 'mean.slope.bp.systolic',
       'num.slope.bp.systolic.visits', 'sum.slope.bp.systolic',
       'first.slope.bp.systolic.date', 'meansquares.slope.bp.systolic',
       'sd.slope.bp.systolic', 'slope.bp.systolic.slope'],
      dtype='object', length=370)
Mean Squared Error: 0.26440007399863213
Predictions saved to ALS_predictions.csv


In [2]:
!pip install rpy2

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import pyreadr
import os

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
pandas2ri.activate()

# --- Prepare RDS Output Using rpy2 ---

# --- Reading and Preparing Data ---
df = pd.read_csv("ALS_progression_rate.csv")

print(df.columns)

data_train = df[df['dFRS'].notna()]
data_predict = df[df['dFRS'].isna()]


print(f"Training dim: {data_train.shape}")
print(f"Prediction dim: {data_predict.shape}")

X = data_train.drop('dFRS', axis=1)
y = data_train['dFRS']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Model Fitting ---
lasso_cv = LassoCV(alphas=None, cv=5, max_iter=10000, random_state=42)
lasso_cv.fit(X_train, y_train)
print(f"Best alpha: {lasso_cv.alpha_}")

y_pred = lasso_cv.predict(X_val)
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
print(f"MSE: {mse:.4f}, RMSE: {rmse:.4f}")

# --- Plotting ---
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.scatter(y_pred, y_val, alpha=0.5)
plt.xlabel("Predicted")
plt.ylabel("Observed")
plt.title("Observed vs Predicted")

residuals = y_val - y_pred
plt.subplot(1, 2, 2)
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(0, color='black', linestyle='--')
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted")
plt.tight_layout()
plt.show()

# --- Prediction for Unlabeled Data ---
X_predict = data_predict.drop('dFRS', axis=1)
predictions = lasso_cv.predict(X_predict)
submission = pd.DataFrame({'predicted': predictions})


os.environ["R_HOME"] = "C:/Program Files/R/R-4.4.2"  

team_name = "team_ashkill"
team_people = ["Shakil", "Ashrif"]
team_error_rate = rmse
team_predictions = submission[['predicted']]  

r_team_people = ro.StrVector(team_people)
r_team_predictions = pandas2ri.py2rpy(team_predictions)
r_list = ro.r['list'](team_name, r_team_people, team_error_rate, r_team_predictions)
output_filename = f"als_progression.{team_name}.rds"


ro.r['saveRDS'](r_list, file=output_filename)
print(f"RDS file saved as {output_filename}")